In [27]:
import pandas as pd
import numpy as np
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')


In [28]:
# 캐나다 달러와 US 달러 통일
train['amount_in_usd'] = train.apply(lambda row: row['total_open_amount'] * 0.75 if row['invoice_currency'] == 'CAD' else row['total_open_amount'], axis=1)
test['amount_in_usd'] = test.apply(lambda row: row['total_open_amount'] * 0.75 if row['invoice_currency'] == 'CAD' else row['total_open_amount'], axis=1)

In [29]:
#total_open_amount의 분포가 매우 치우쳐져 있기 때문에, 로그 변환을 통해 분포 완화
train['amount_in_usd'] = np.log1p(train['amount_in_usd'])
test['amount_in_usd'] = np.log1p(test['amount_in_usd'])

In [30]:
# 결제 수단의 등장 횟수가 30회 이하인 경우 Other 클래스로 변환
counts = train["cust_payment_terms"].value_counts()
train["cust_payment_terms_grp"] = train["cust_payment_terms"].where(
    train["cust_payment_terms"].map(counts) > 30,
    "Other"
)
test["cust_payment_terms_grp"] = test["cust_payment_terms"].where(
    test["cust_payment_terms"].map(counts) > 30,
    "Other"
)

In [31]:
# 송장이 생성된 기준 날짜의 월, 일, 요일 추출
train['baseline_create_date'] = pd.to_datetime(train['baseline_create_date'], format='%Y%m%d', errors='coerce')
test['baseline_create_date'] = pd.to_datetime(test['baseline_create_date'], format='%Y%m%d', errors='coerce')

train['baseline_month'] = train['baseline_create_date'].dt.month
train['baseline_day'] = train['baseline_create_date'].dt.day
train['baseline_dayofweek'] = train['baseline_create_date'].dt.dayofweek
test['baseline_month'] = test['baseline_create_date'].dt.month
test['baseline_day'] = test['baseline_create_date'].dt.day
test['baseline_dayofweek'] = test['baseline_create_date'].dt.dayofweek


In [32]:
# 송장 생성일과 마감일 사이의 기간 추가
train['due_in_date'] = pd.to_datetime(train['due_in_date'], errors='coerce')
test['due_in_date'] = pd.to_datetime(test['due_in_date'], errors='coerce')
train['Allowed_Pay_Days'] = (train['due_in_date'] - train['baseline_create_date']).dt.days
test['Allowed_Pay_Days'] = (test['due_in_date'] - test['baseline_create_date']).dt.days

In [34]:
# cust_number 자릿수 맞추기
def clean_cust_number(df):
    df = df.copy()

    cust_number_stripped = df["cust_number"].astype(str).str.strip()
    is_9_digit = cust_number_stripped.str.fullmatch(r"\d{9}")

    df["cust_number"] = cust_number_stripped.where(
        ~is_9_digit,
        cust_number_stripped.str.zfill(10)
    )

    return df

train = clean_cust_number(train)
test = clean_cust_number(test)

In [35]:
drop_cols = ['doc_id', 'invoice_currency', 'document type', 'area_business', 'isOpen', 'invoice_id','document_create_date', 'document_create_date.1', 'posting_date', 'total_open_amount']
train.drop(columns = drop_cols, inplace=True)
test.drop(columns = drop_cols, inplace=True)


In [36]:
train.to_csv('../data/train_cleaned.csv', index=False)
test.to_csv('../data/test_cleaned.csv', index=False)

In [37]:
train

,business_code,cust_number,name_customer,clear_date,buisness_year,due_in_date,posting_id,baseline_create_date,cust_payment_terms,target,amount_in_usd,cust_payment_terms_grp,baseline_month,baseline_day,baseline_dayofweek,Allowed_Pay_Days
0,U001,0200706844,WINC co,2019-02-19,2019.0,2018-12-29,1.0,2018-12-14,NAA8,1,6.359366,NAA8,12,14,4,15
1,CA02,0140104249,SOB associates,2019-01-23,2019.0,2018-12-24,1.0,2018-12-14,CA10,1,11.663576,CA10,12,14,4,10
2,U001,0200769623,WAL-MAR systems,2019-01-09,2019.0,2019-01-14,1.0,2018-12-30,NAH4,0,8.197049,NAH4,12,30,6,15
3,U001,0200769623,WAL-MAR foundation,2019-01-09,2019.0,2019-01-14,1.0,2018-12-30,NAH4,0,9.428658,NAH4,12,30,6,15
4,U001,0200792734,MDV/ corp,2019-01-14,2019.0,2019-01-14,1.0,2018-12-30,NAA8,0,11.457984,NAA8,12,30,6,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31995,U001,0200764795,SYSCO trust,2019-12-04,2019.0,2019-12-03,1.0,2019-11-18,NAA8,1,7.709604,NAA8,11,18,0,15
31996,U001,0200772670,ASSOCIAT co,2019-12-04,2019.0,2019-12-03,1.0,2019-11-18,NAU5,1,8.570087,NAU5,11,18,0,15
31997,U001,0200769623,WAL-MAR associates,2019-11-29,2019.0,2019-12-03,1.0,2019-11-18,NAH4,0,11.013749,NAH4,11,18,0,15
31998,CA02,0140104429,COSTCO co,2019-12-03,2019.0,2019-11-28,1.0,2019-11-18,CA10,1,11.514690,CA10,11,18,0,10
